## Import library

In [1]:
# 0. Setup
import os, json, re, textwrap
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any
from dotenv import load_dotenv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# ML
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    roc_auc_score, f1_score, roc_curve,
    mean_absolute_percentage_error, mean_absolute_error, r2_score,
)
from xgboost import XGBRegressor

# RAG / PDF / vector store
import pypdf
import psycopg2
from sentence_transformers import SentenceTransformer
from sqlalchemy import create_engine, text

# LLM providers
from huggingface_hub import InferenceClient
from groq import Groq

## Configuration

In [2]:

# ---- Paths (cwd = data_science/notebooks/) ----
BASE_PATH = Path.cwd().resolve().parents[1]
PROJECT   = BASE_PATH / "credit_risk_production"
DATA_PATH = PROJECT / "database" / "data" / "merged_credit_risk_data.parquet"
PDF_DIR   = PROJECT / "database" / "pdf"
MODEL_DIR = PROJECT / "models" / "credit_risk"
ML_DIR    = MODEL_DIR / "ml_credit_risk"
META_PATH = MODEL_DIR / "metadata_credit_risk" / "metadata.json"
ENV_PATH = BASE_PATH / "data_science" / "superset" / ".env"
MODEL_BUNDLE = PROJECT / "database" / "LLM" / "outputs_llm" / "model_artifacts" / "model_bundle.joblib"

REPORTS   = BASE_PATH / "data_science" / "reports"
REPORTS.mkdir(exist_ok=True, parents=True)
load_dotenv(ENV_PATH)  # Load environment variables from .env file

# ---- Postgres + pgvector ----
POSTGRES_HOST = "localhost"
POSTGRES_PORT = os.getenv("POSTGRES_PORT")
POSTGRES_USER = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_DB = os.getenv("POSTGRES_DB")

PG_URL = f"postgresql+psycopg2://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}"
engine = create_engine(PG_URL, pool_pre_ping=True)

# ---- LLM clients (HF primary, Groq fallback) ----
HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")
GROQ_API_KEY        = os.getenv("GROQ_API_KEY")

hf_client   = InferenceClient(api_key=HUGGINGFACE_API_KEY) if HUGGINGFACE_API_KEY else None
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

HF_MODEL   = os.getenv("HF_MODEL", "Qwen/Qwen2.5-Coder-32B-Instruct")
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")

# ---- Embedder ----
EMBED_MODEL = os.getenv("EMBED_MODEL", "BAAI/bge-small-en-v1.5")
embedder = SentenceTransformer(EMBED_MODEL)

# ---- Domain constants ----
TARGET, ID_COL = "Approved_Flag", "PROSPECTID"
META = json.loads(META_PATH.read_text())
FEATURES, CLASS_LABELS = META["feature_columns"], META["class_labels"]

print(f"pdfs: {[p.name for p in PDF_DIR.glob('*.pdf')]}")
print(f"hf_client: {bool(hf_client)}  groq_client: {bool(groq_client)}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

pdfs: ['Fraud_Typologies_and_Red Flags .pdf', 'Regulatory_Risk Policy Core .pdf', 'Delinquency_Classification .pdf', 'Scorecard_Cut-off Policy.pdf']
hf_client: True  groq_client: True


In [3]:
with open(MODEL_BUNDLE, "rb") as f:
    model_bundle = joblib.load(f)
    print("loaded model bundle:", list(model_bundle.keys()))

loaded model bundle: ['models', 'scaler', 'label_encoders', 'feature_columns', 'class_labels']


## Setup LLM Chat

In [4]:
# 1. Unified LLM client with fallback
def llm_chat(messages: List[Dict[str, str]],
             max_tokens: int = 768,
             temperature: float = 0.3) -> Dict[str, Any]:
    """Try HF first; fall back to Groq. Returns {'text', 'provider'}."""
    if hf_client:
        try:
            resp = hf_client.chat.completions.create(
                model=HF_MODEL,
                messages=messages,
                max_tokens=max_tokens,
                temperature=temperature,
            )
            return {"text": resp.choices[0].message.content, "provider": "huggingface"}
        except Exception as e:
            err = str(e)
            if "402" in err or "depleted" in err:
                print("⚠️  HF quota depleted — falling back to Groq")
            else:
                print(f"❌  HF failed: {err}")

    if groq_client:
        try:
            resp = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=messages,
                max_tokens=max_tokens,
                temperature=temperature,
            )
            return {"text": resp.choices[0].message.content, "provider": "groq"}
        except Exception as e:
            print(f"❌  Groq failed: {str(e)}")

    raise RuntimeError("No LLM provider available.")

# smoke test
out = llm_chat([{"role": "user", "content": "Reply with the single word: OK"}], max_tokens=10)
print(out)

{'text': 'OK', 'provider': 'huggingface'}


## ML Layer

In [5]:
# Load database
df = pd.read_parquet(DATA_PATH)
print(f"Rows={len(df):,} cols={len(df.columns)}")
df.head()

Rows=51,336 cols=87


,PROSPECTID,Total_TL,Tot_Closed_TL,Tot_Active_TL,Total_TL_opened_L6M,Tot_TL_closed_L6M,pct_tl_open_L6M,pct_tl_closed_L6M,pct_active_tl,pct_closed_tl,...,pct_CC_enq_L6m_of_L12m,pct_PL_enq_L6m_of_ever,pct_CC_enq_L6m_of_ever,max_unsec_exposure_inPct,HL_Flag,GL_Flag,last_prod_enq2,first_prod_enq2,Credit_Score,Approved_Flag
0,1,5,4,1,0,0,0.000,0.0,0.200,0.800,...,0.0,0.0,0.0,13.333,1,0,PL,PL,696,P2
1,2,1,0,1,0,0,0.000,0.0,1.000,0.000,...,0.0,0.0,0.0,0.860,0,0,ConsumerLoan,ConsumerLoan,685,P2
2,3,8,0,8,1,0,0.125,0.0,1.000,0.000,...,0.0,0.0,0.0,5741.667,1,0,ConsumerLoan,others,693,P2
3,4,1,0,1,1,0,1.000,0.0,1.000,0.000,...,0.0,0.0,0.0,9.900,0,0,others,others,673,P2
4,5,3,2,1,0,0,0.000,0.0,0.333,0.667,...,0.0,0.0,0.0,-99999.000,0,0,AL,AL,753,P1


In [6]:
# Target distribution
base = (df[TARGET].value_counts(normalize=True) * 100).round(2).rename("pct")
display(base.reset_index().rename(columns={"index": TARGET}))

,Approved_Flag,pct
0,P2,62.72
1,P3,14.52
2,P4,11.46
3,P1,11.30


## Encode features + target

In [7]:
X = df[FEATURES].copy()
y_raw = df[TARGET].astype(str)

le_map = {}
for col in X.select_dtypes(include=["object", "category"]):
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    le_map[col] = le

# Encode target
target_le = LabelEncoder()
y = target_le.fit_transform(y_raw)
CLASSES = list(target_le.classes_)
print(f"Classes: {CLASSES}")

Classes: ['P1', 'P2', 'P3', 'P4']


## Split data + Load models

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

MODELS = {
    "Logistic Regression": joblib.load(ML_DIR / "logistic_regression.joblib"),
    "Random Forest":       joblib.load(ML_DIR / "random_forest.joblib"),
    "Gradient Boosting":   joblib.load(ML_DIR / "gradient_boosting.joblib"),
    "XGBoost":             joblib.load(ML_DIR / "xgboost.joblib"),
    "K-Nearest Neighbors": joblib.load(ML_DIR / "k_nearest_neighbors.joblib"),
    "Decision Tree":       joblib.load(ML_DIR / "decision_tree.joblib"),
}
print("loaded:", list(MODELS.keys()))

loaded: ['Logistic Regression', 'Random Forest', 'Gradient Boosting', 'XGBoost', 'K-Nearest Neighbors', 'Decision Tree']


## Scale data

In [9]:
# Scale the X_train and X_test using the scaler from the model bundle
scaler = model_bundle['scaler']
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Data after scaling: X train scaled={X_train_scaled.shape}, X test scaled={X_test_scaled.shape}")

Data after scaling: X train scaled=(41068, 47), X test scaled=(10268, 47)


In [10]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

# Score metrics for every model
rows = []
for name, model in MODELS.items():
    p = model.predict_proba(X_test_scaled)
    pred = p.argmax(axis=1)

    # Append metrics
    rows.append({
        "model": name,
        "roc_auc": roc_auc_score(y_test, p, multi_class="ovr", average="weighted"),
        "f1_weighted": f1_score(y_test, pred, average="weighted")
    })

scoreboard = pd.DataFrame(rows).sort_values(by="roc_auc", ascending=False).reset_index(drop=True)
display(scoreboard)

# Define champion model
CHAMPION = scoreboard.iloc[0]["model"]
print(f"🏆 Champion model: {CHAMPION}")

,model,roc_auc,f1_weighted
0,Logistic Regression,0.971802,0.912988
1,Random Forest,0.967283,0.942339
2,Gradient Boosting,0.949901,0.935937
3,Decision Tree,0.945281,0.937560
4,XGBoost,0.940799,0.928983
5,K-Nearest Neighbors,0.887665,0.720734


🏆 Champion model: Logistic Regression


## Monitoring metrics - Gini/KS/PSI

In [11]:
def gini(y_bin, score):
    return 2 * roc_auc_score(y_bin, score) - 1

def ks(y_bin, score):
    fpr, tpr, _ = roc_curve(y_bin, score)
    return float(np.max(np.abs(tpr - fpr)))

def psi(ref, cur, bins=10):
    q = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))

    if len(q) < 3:
        return 0.0

    r, _ = np.histogram(ref, bins=q)
    c, _ = np.histogram(cur, bins=q)
    r = np.clip(r / max(r.sum(), 1), 1e-6, None)
    c = np.clip(c / max(c.sum(), 1), 1e-6, None)
    return float(np.sum((r - c) * np.log(r / c)))

# Class labels
CLS = CLASS_LABELS[-1]
y_bin_test = (y_test == CLS).astype(int)
y_bin_train = (y_train == CLS).astype(int)

# Metrics rows for evaluation
metrics_rows = []
for name, model in MODELS.items():
    p_train = model.predict_proba(X_train_scaled)[:, CLS]
    p_test = model.predict_proba(X_test_scaled)[:, CLS]
    
    # model.predict() directly gives 1D class predictions
    preds_test = model.predict(X_test_scaled)
    
    metrics_rows.append({
        "model": name,
        "gini": gini(y_bin_test, p_test),
        "ks": ks(y_bin_test, p_test),
        "psi": psi(p_train, p_test),
        "roc_auc": roc_auc_score(y_test, model.predict_proba(X_test_scaled), multi_class="ovr", average="weighted"),
        "f1_weighted": f1_score(y_test, preds_test, average="weighted"),
    })

# Create metrics DataFrame
metrics_df = pd.DataFrame(metrics_rows).sort_values(by="gini", ascending=False).reset_index(drop=True)
display(metrics_df.round(4))

,model,gini,ks,psi,roc_auc,f1_weighted
0,Logistic Regression,0.9998,0.9999,0.0002,0.9718,0.9130
1,Random Forest,0.9866,0.8959,0.0017,0.9673,0.9423
2,XGBoost,0.9393,0.8341,0.0001,0.9408,0.9290
3,Gradient Boosting,0.9385,0.8341,0.0001,0.9499,0.9359
4,K-Nearest Neighbors,0.8404,0.7252,0.0037,0.8877,0.7207
5,Decision Tree,0.8341,0.8341,0.0000,0.9453,0.9376


## Income model

In [27]:
INC = "NETMONTHLYINCOME"
inc_feats = [c for c in FEATURES if c != INC]

d2 = df[[INC] + inc_feats].copy()
for c in d2.select_dtypes(include=["object", "category"]):
    d2[c] = LabelEncoder().fit_transform(d2[c].astype(str))

Xi, yi = d2[inc_feats], d2[INC].astype(float)
mask = yi > 0
Xi, yi = Xi[mask], yi[mask]
Xi_train, Xi_test, yi_train, yi_test = train_test_split(Xi, yi, test_size=0.2, random_state=42)

# Train XGBoost regressor as income model
income_model = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    tree_method="hist",
)
income_model.fit(Xi_train, yi_train)
y_hat = income_model.predict(Xi_test)

income_metrics = {
    "mape": mean_absolute_percentage_error(yi_test, y_hat),
    "mae": mean_absolute_error(yi_test, y_hat),
    "r2": r2_score(yi_test, y_hat),
}

# Insert values
income_metrics["baseline_mape"] = mean_absolute_percentage_error(yi_test, np.full_like(yi_test, yi_train.mean()))
income_metrics["improvement_vs_naive"] = 1 - income_metrics["mape"] / income_metrics["baseline_mape"]
income_metrics

{'mape': 11.112313112539828,
 'mae': 8818.107290082378,
 'r2': 0.09760719378895488,
 'baseline_mape': 14.08032974264458,
 'improvement_vs_naive': 0.2107916990832699}

## Build the deterministic facts payload

In [28]:
FACTS = {
    "as_of": datetime.now().isoformat(timespec='seconds'),
    "rows_analyzed": int(len(df)),
    "class_distribution": {str(k): int(v) for k, v in y_raw.value_counts().items()},
    "champion": CHAMPION,
    "model_metrics": metrics_df.round(4).to_dict(orient="records"),
    "income_model": {k: round(v, 4) for k, v in income_metrics.items()},
    "thresholds": {"gini_min": 0.30, "ks_min": 0.20, "psi_max": 0.25},
}
print(json.dumps(FACTS, indent=2)[:1200], "...")

{
  "as_of": "2026-09-21T14:43:49",
  "rows_analyzed": 51336,
  "class_distribution": {
    "P2": 32199,
    "P3": 7452,
    "P4": 5882,
    "P1": 5803
  },
  "champion": "Logistic Regression",
  "model_metrics": [
    {
      "model": "Logistic Regression",
      "gini": 0.9998,
      "ks": 0.9999,
      "psi": 0.0002,
      "roc_auc": 0.9718,
      "f1_weighted": 0.913
    },
    {
      "model": "Random Forest",
      "gini": 0.9866,
      "ks": 0.8959,
      "psi": 0.0017,
      "roc_auc": 0.9673,
      "f1_weighted": 0.9423
    },
    {
      "model": "XGBoost",
      "gini": 0.9393,
      "ks": 0.8341,
      "psi": 0.0001,
      "roc_auc": 0.9408,
      "f1_weighted": 0.929
    },
    {
      "model": "Gradient Boosting",
      "gini": 0.9385,
      "ks": 0.8341,
      "psi": 0.0001,
      "roc_auc": 0.9499,
      "f1_weighted": 0.9359
    },
    {
      "model": "K-Nearest Neighbors",
      "gini": 0.8404,
      "ks": 0.7252,
      "psi": 0.0037,
      "roc_auc": 0.8877,
      "

## ---- PART B ----

## RAG over the PDFs

In [29]:
# B1. Read all PDFs under database/pdf/
def read_pdf(path: Path) -> List[Dict]:
    reader = pypdf.PdfReader(str(path))
    pages = []
    for i, page in enumerate(reader.pages):
        txt = (page.extract_text() or "").strip()
        if txt:
            pages.append({"doc": path.name, "page": i + 1, "text": txt})
    return pages

docs = []
for pdf in sorted(PDF_DIR.glob("*.pdf")):
    docs.extend(read_pdf(pdf))

print(f"pages: {len(docs)}  docs: {sorted({d['doc'] for d in docs})}")

pages: 291  docs: ['Delinquency_Classification .pdf', 'Fraud_Typologies_and_Red Flags .pdf', 'Regulatory_Risk Policy Core .pdf', 'Scorecard_Cut-off Policy.pdf']


## Chunk with overlap

In [30]:
def chunk(text: str, size: int = 800, overlap: int = 100) -> List[str]:
    """Chunk text into overlapping segments of specified size"""
    out, s = [], 0
    while s < len(text):
        e = min(s + size, len(text))
        out.append(text[s:e])
        s = e - overlap if e < len(text) else e

    return out

# Define chunks
chunks = []
for d in docs:
    for i, c in enumerate(chunk(d["text"])):
        chunks.append({
            "chunk_id": f"{d['doc']}::p{d['page']}::c{i}",
            "doc": d["doc"], "page": d["page"], "text": c,
        })
print(f"chunks: {len(chunks)}")

chunks: 1125


## Index into PGVector

In [31]:
# B3. Postgres + pgvector index
with engine.begin() as conn:
    conn.execute(text("CREATE EXTENSION IF NOT EXISTS vector;"))
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS credit_risk.policy_chunks (
            chunk_id  TEXT PRIMARY KEY,
            doc       TEXT NOT NULL,
            page      INT  NOT NULL,
            text      TEXT NOT NULL,
            embedding vector(384)
        );
        CREATE INDEX IF NOT EXISTS policy_chunks_embedding_idx
            ON credit_risk.policy_chunks
            USING ivfflat (embedding vector_cosine_ops) WITH (lists = 100);
    """))

embeddings = embedder.encode(
    [c["text"] for c in chunks],
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True,
)

raw = psycopg2.connect(PG_URL.replace("postgresql+psycopg2://", "postgresql://"))
with raw, raw.cursor() as cur:
    cur.execute("TRUNCATE credit_risk.policy_chunks;")
    for c, e in zip(chunks, embeddings):
        cur.execute(
            "INSERT INTO credit_risk.policy_chunks "
            "(chunk_id, doc, page, text, embedding) VALUES (%s,%s,%s,%s,%s)",
            (c["chunk_id"], c["doc"], c["page"], c["text"], e.tolist()),
        )
print(f"indexed {len(chunks)} chunks")

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

indexed 1125 chunks


## Retrieval with citations

In [32]:
def retrieve(query: str, k: int = 5) -> List[Dict]:
    """Retrieve top-k relevant chunks from Postgres + pgvector"""
    q = embedder.encode([query], normalize_embeddings=True)[0].tolist()
    sql = text("""
        SELECT doc, page, chunk_id, text,
               1 - (embedding <=> CAST(:q AS vector)) AS score
        FROM credit_risk.policy_chunks
        ORDER BY embedding <=> CAST(:q AS vector)
        LIMIT :k
    """)
    with engine.connect() as conn:
        return [dict(r) for r in conn.execute(sql, {"q": q, "k": k}).mappings()]

# Sanity check
query = "what is impact to customer if the loan is rejected about risk scoring and credit risk?"
for r in retrieve(query, k=3):
    print(f"[{r['score']:.3f}] {r['doc']} p{r['page']}")
    print(textwrap.shorten(r['text'], width=200), "\n---")

print(50*"=")
print("✅  Postgres + pgvector retrieval works")

[0.719] Scorecard_Cut-off Policy.pdf p25
Introduction > Risk Management Comptroller’s Handbook 22 Credit Card Lending • Scoring accuracy reports: Present the volume of scoring errors sorted by those deemed significant versus minor. [...] 
---
[0.687] Delinquency_Classification .pdf p9
other words, such an asset will have well defined credit weaknesses that jeopardise the liquidation of the debt and are characterised by the distinct possibility that the banks will sustain some [...] 
---
[0.675] Scorecard_Cut-off Policy.pdf p8
rank Wall Street Reform and Consumer Protection Act (Dodd–Frank Act), 12 USC 5536(a), which prohibits unfair, deceptive, or abusive acts or practices (UDAAP). Marketing materials for promotional [...] 
---
✅  Postgres + pgvector retrieval works


## ---- PART C ----

## LLM Explanation Layer

### Build prompt template

In [33]:
system_prompt = """
You are a senior credit-risk analyst writing report for a bank's & fintech.
Rules:
1. Quote a numeric value from FACTS for every claim.
2. If you invoke a policy, cite it as [DOC pN] and only use text from POLICY.
3. End each item with a business implication (NPF, FPD30, approval rate, cost).
4. Never invent numbers or policy. If missing, say "not available" and explain why shortly.
5. Output strict JSON:
{
  "gini":  "...",
  "ks":    "...",
  "psi":   "...",
  "summary": "...",
  "actions": ["...", "...", "..."]
}"""

def user_prompt(facts: dict, policy_chunks: List[dict], question: str) -> str:
    policy_block = "\n\n".join(
        f"[{c['doc']} p{c['page']}] {c['text']}" for c in policy_chunks
    )
    return f"""QUESTION
    {question}

    FACTS (only numbers you may use)
    {json.dumps(facts, indent=2)}
    
    POLICY (only text you may cite)
    {policy_block}

    Write the JSON answer now.
    """

### LLM call for the explanation & Parse JSON answer

In [34]:
QUESTION = ("explain this month's model health: Gini, KS, and PSI for the champion model,"
            "whether anything breaches policy, and recommended actions")

policy_chunks = retrieve(QUESTION, k=6)
response = llm_chat([
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt(FACTS, policy_chunks, QUESTION)}
], max_tokens=512, temperature=0.3)

# Parse JSON robustly
import re

def parse_json(text_or_dict, provider=None):
    # If a dict was passed (e.g. {"text": ..., "provider": ...})
    if isinstance(text_or_dict, dict):
        provider = text_or_dict.get("provider", provider)
        raw_text = text_or_dict.get("text", "")
    else:
        raw_text = text_or_dict

    parsed_result = None

    if isinstance(raw_text, str):
        # Remove markdown code fences like ```json ... ```
        cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_text.strip(), flags=re.MULTILINE)
        try:
            parsed_result = json.loads(cleaned)
        except Exception:
            # Fallback regex match for JSON object
            match = re.search(r"\{.*\}", cleaned, re.DOTALL)
            if match:
                try:
                    parsed_result = json.loads(match.group(0))
                except Exception:
                    pass
            
            if parsed_result is None:
                parsed_result = {"raw": raw_text}
    elif isinstance(raw_text, dict):
        parsed_result = raw_text
    else:
        parsed_result = {"raw": str(raw_text)}

    # Attach provider to output
    if provider:
        if isinstance(parsed_result, dict):
            parsed_result["provider"] = provider
        else:
            parsed_result = {"data": parsed_result, "provider": provider}

    return parsed_result

# Call narrative as JSON parsed
narrative = parse_json({"text": response["text"], "provider": response["provider"]})
print(json.dumps(narrative, indent=2)[:1500])

{
  "gini": "0.9998",
  "ks": "0.9999",
  "psi": "0.0002",
  "summary": "This month's champion model, Logistic Regression, exhibits excellent performance with a Gini coefficient of 0.9998, indicating strong separation between good and bad risks. The KS score of 0.9999 further confirms its superior ability to discriminate between the two classes. The PSI value of 0.0002 suggests minimal changes in the distribution of the population, maintaining the model's stability. None of these metrics breach the specified thresholds (Gini min: 0.3, KS min: 0.2, PSI max: 0.25).",
  "actions": [
    "Continue monitoring the Logistic Regression model to ensure it maintains high performance and stability [Scorecard_Cut-off Policy.pdf p118].",
    "Ensure that any automated processes related to this model comply with the bank's credit card policies and regulatory requirements [Scorecard_Cut-off Policy.pdf p89].",
    "Periodically review the classification practices of accounts to ensure consistency with

## --- PART D ---

### Build Agent tools for Agentic AI

In [35]:
def tool_get_metrics() -> dict:
    return FACTS

def tool_check_thresholds(facts: dict) -> list[dict]:
    t = facts["thresholds"]
    out = []
    for m in facts["model_metrics"]:
        # Append a dict into model_metrics with threshold checks
        out.append({
            "model": m["model"],
            "gini": m["gini"],
            "gini_ok": m["gini"] >= t["gini_min"],
            "ks": m["ks"],
            "ks_ok": m["ks"] >= t["ks_min"],
            "psi": m["psi"],
            "psi_ok": m["psi"] <= t["psi_max"]
        })

    return out

def tool_retrieve_policy(query: str, k: int = 6) -> list[dict]:
    return retrieve(query, k=k)

TOOLS = {
    "get_metics": tool_get_metrics,
    "check_thresholds": tool_check_thresholds,
    "retrieve_policy": tool_retrieve_policy,
}

### Agent Loop
* ### Agent: plan -> run tools -> LLM explanation -> assemble -> audit

In [36]:
def agent_run(question: str) -> dict:
    # Build list to storage trace of tool calls
    trace = []

    facts = TOOLS["get_metics"]()
    trace.append("get_metrics")

    checks = TOOLS["check_thresholds"](facts)
    trace.append("check_thresholds")

    policy = TOOLS["retrieve_policy"](question, k=5)
    trace.append("retrieve_policy")

    response = llm_chat([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt(facts, policy, question)}
    ], max_tokens=512, temperature=0.3)
    trace.append("llm_chat")

    # Create narrative with provider info
    narrative = parse_json({"text": response["text"]})

    return {
        "as_of": facts["as_of"],
        "question": question,
        "facts": facts,
        "checks": checks,
        "policy_citations": [{"doc": c["doc"], "page": c["page"],
                              "score": round(c["score"], 3)} for c in policy],
        "narrative": narrative,
        "provider": response["provider"],
        "trace": trace
    }

# report results
AGENT_QUESTION = "borrowers approval for borrowers based on credit risk analysis and model health"
report = agent_run(AGENT_QUESTION)
print(json.dumps(report["narrative"], indent=2)[:1500])

{
  "gini": "0.9998",
  "ks": "0.9999",
  "psi": "0.0002",
  "summary": "The Logistic Regression model, which is the champion model, exhibits excellent performance with a Gini coefficient of 0.9998, a KS statistic of 0.9999, and a PSI of 0.0002. These metrics indicate that the model effectively discriminates between good and bad borrowers and has minimal divergence from the training data. The model's performance significantly outperforms the thresholds set (Gini min: 0.3, KS min: 0.2, PSI max: 0.25). No other model comes close to these performance levels.",
  "actions": [
    "Continue using the Logistic Regression model for borrower approvals as it meets all the performance criteria and is the most effective model among those analyzed [Scorecard_Cut-off Policy.pdf p63].",
    "Ensure that the model is regularly monitored and validated to maintain its accuracy and relevance [Scorecard_Cut-off Policy.pdf p23].",
    "Review and document any changes in cutoff scores and their implication

In [37]:
report["trace"]

['get_metrics', 'check_thresholds', 'retrieve_policy', 'llm_chat']

## Guardrail: no invented numbers

In [38]:
def nums(text: str) -> set:
    return set(re.findall(r"\d+\.\d+", text))

# Implement a function to extract
allowed = nums(json.dumps(report["facts"]))
used = nums(json.dumps(report["narrative"]))
invented = used - allowed

print(f"allowed={len(allowed)} used={len(used)}  invented={len(invented)}")

allowed=34 used=6  invented=0


### Render markdown report

In [39]:
# D4. Render the final report
def render_md(r: dict) -> str:
    n, f = r["narrative"], r["facts"]
    lines = [
        f"# Credit Risk — Model Health Report ({r['as_of']})",
        f"**Champion:** {f['champion']}  |  **Rows:** {f['rows_analyzed']:,}  |  **Provider:** {r['provider']}",
        "",
        "## Metrics",
        "| Model | Gini | KS | PSI | ROC AUC | F1 |",
        "|---|---|---|---|---|---|",
    ]
    for m in f["model_metrics"]:
        lines.append(
            f"| {m['model']} | {m['gini']:.4f} | {m['ks']:.4f} | {m['psi']:.4f} | "
            f"{m['roc_auc']:.4f} | {m['f1_weighted']:.4f} |"
        )
    lines += [
        "", "## Threshold checks",
        "| Model | Gini OK | KS OK | PSI OK |", "|---|---|---|---|",
    ]
    for c in r["checks"]:
        lines.append(f"| {c['model']} | {c['gini_ok']} | {c['ks_ok']} | {c['psi_ok']} |")
    lines += [
        "", "## Interpretation",
        f"- **Gini:** {n.get('gini','—')}",
        f"- **KS:** {n.get('ks','—')}",
        f"- **PSI:** {n.get('psi','—')}",
        "",
        f"**Summary.** {n.get('summary','—')}",
        "", "## Actions", *[f"- {a}" for a in n.get("actions", [])],
        "", "## Sources",
        *[f"- {c['doc']} p{c['page']} (score {c['score']})" for c in r["policy_citations"]],
        "", f"_trace: {', '.join(r['trace'])}_",
    ]
    return "\n".join(lines)

md = render_md(report)
out = REPORTS / f"credit_risk_report_{datetime.utcnow():%Y-%m}.md"
out.write_text(md)
print("wrote:", out)

wrote: /Users/miftahhadiyannoor/Documents/credit_risk/data_science/reports/credit_risk_report_2026-09.md


### Audit log

In [40]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS credit_risk.llm_report_runs (
            run_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
            payload JSONB NOT NULL
        );
    """))
    conn.execute(
        text("INSERT INTO credit_risk.llm_report_runs (payload) VALUES (:p)"),
        {"p": json.dumps(report)}
    )

print("✅  Report run logged to Postgres")

✅  Report run logged to Postgres
